In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
import os, time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
import warnings
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Kutuphaneler yuklendi.")

In [ ]:
# --- KONFIGURASYON (MobileNetV3-Large + CutMix + Rotation Aug) ---
MODEL_NAME = 'mobilenetv3_large_100'
EXPERIMENT_NAME = "MobileNetV3Large_CutMixAug_R1"

BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.0002
NUM_CLASSES = 8
DROPOUT_RATE = 0.5
WEIGHT_DECAY = 0.0005
LABEL_SMOOTHING = 0.1
INPUT_SIZE = 224
USE_CLASS_WEIGHTED_LOSS = False
FORCE_WEIGHTED_SAMPLER = False
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_METRIC = 'val_macro_f1'
NUM_WORKERS = 2 if os.name == 'nt' else 4

CUTMIX_ALPHA = 1.0
CUTMIX_PROB  = 0.5

dropout_tag = f"dropout{int(round(DROPOUT_RATE * 10)):02d}"
ls_tag      = f"ls{int(round(LABEL_SMOOTHING * 100)):02d}"
wd_tag      = f"wd{str(WEIGHT_DECAY).replace('.', 'p')}"
lr_tag      = f"lr{LEARNING_RATE:g}".replace('.', 'p')
sampler_tag = "smpON" if FORCE_WEIGHTED_SAMPLER else "smpOFF"
cwl_tag     = "cwlON" if USE_CLASS_WEIGHTED_LOSS else "cwlOFF"
RUN_TAG  = f"{dropout_tag}_{ls_tag}_{wd_tag}_bs{BATCH_SIZE}_ep{EPOCHS}_{lr_tag}_i{INPUT_SIZE}_cutmixAug_{sampler_tag}_{cwl_tag}"
RUN_NAME = f"{EXPERIMENT_NAME}_{RUN_TAG}"

cwd = os.getcwd()
PROJECT_ROOT = cwd if os.path.isdir(os.path.join(cwd, 'data', 'prepared-data')) \
               else os.path.abspath(os.path.join(cwd, '..'))

DATA_DIR          = os.path.join(PROJECT_ROOT, 'data', 'prepared-data')
OUTPUT_DIR        = os.path.join(PROJECT_ROOT, 'models', 'pytorch', RUN_NAME)
RESULT_OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs', MODEL_NAME)
PLOTS_DIR         = os.path.join(RESULT_OUTPUT_DIR, 'plots', 'cutmix_aug')
REPORTS_DIR       = os.path.join(RESULT_OUTPUT_DIR, 'reports')
BEST_MODEL_PATH   = os.path.join(OUTPUT_DIR, f'best_{MODEL_NAME}.pth')

for d in [OUTPUT_DIR, PLOTS_DIR, REPORTS_DIR]: os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cihaz: {DEVICE}  Model: {MODEL_NAME}')
print(f'CutMix Alpha: {CUTMIX_ALPHA}, Prob: {CUTMIX_PROB}')

In [ ]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.15),
        transforms.RandomRotation(degrees=15),
        transforms.RandomApply([
            transforms.ColorJitter(brightness=0.1, contrast=0.12, saturation=0.08, hue=0.02)
        ], p=0.4),
        transforms.RandomAutocontrast(p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val':  transforms.Compose([transforms.Resize((INPUT_SIZE, INPUT_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)]),
    'test': transforms.Compose([transforms.Resize((INPUT_SIZE, INPUT_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x]) for x in ['train', 'val', 'test']}
train_targets  = np.array(image_datasets['train'].targets)
class_counts   = np.bincount(train_targets, minlength=NUM_CLASSES)
class_weights  = class_counts.sum() / (NUM_CLASSES * np.maximum(class_counts, 1))
CLASS_WEIGHTS_TENSOR = torch.tensor(class_weights, dtype=torch.float32)

train_sampler = None
loader_kw = {'num_workers': NUM_WORKERS, 'pin_memory': (DEVICE == 'cuda')}
if NUM_WORKERS > 0: loader_kw.update({'persistent_workers': True, 'prefetch_factor': 2})

dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, **loader_kw),
    'val':   DataLoader(image_datasets['val'],   batch_size=BATCH_SIZE, shuffle=False, **loader_kw),
    'test':  DataLoader(image_datasets['test'],  batch_size=BATCH_SIZE, shuffle=False, **loader_kw)
}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names   = image_datasets['train'].classes
print(f'Siniflar: {class_names}')
print(f'Egitim: {dataset_sizes["train"]}  Val: {dataset_sizes["val"]}  Test: {dataset_sizes["test"]}')

In [ ]:
def cutmix_data(inputs, targets, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(inputs.size(0), device=inputs.device)
    ta, tb = targets, targets[idx]
    _, _, H, W = inputs.shape
    cut_w, cut_h = int(W * np.sqrt(1 - lam)), int(H * np.sqrt(1 - lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
    x2, y2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
    inputs[:, :, y1:y2, x1:x2] = inputs[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return inputs, ta, tb, lam

def cutmix_criterion(crit, out, ta, tb, lam):
    return lam * crit(out, ta) + (1 - lam) * crit(out, tb)

print('CutMix tanimlandi.')

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES, drop_rate=DROPOUT_RATE).to(DEVICE)
loss_weight = CLASS_WEIGHTS_TENSOR.to(DEVICE) if USE_CLASS_WEIGHTED_LOSS else None
criterion   = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING, weight=loss_weight)
optimizer   = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler   = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.3, patience=4, verbose=True)
print(f'Parametre: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()
    best_acc = best_val_macro_f1 = 0.0
    best_val_loss = float('inf')
    best_metric = -float('inf')
    no_improve  = 0
    history = {'train_loss':[], 'train_acc':[], 'train_macro_f1':[],
                'val_loss':[],  'val_acc':[],   'val_macro_f1':[], 'lr':[]}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}'); print('-'*10)
        for phase in ['train','val']:
            model.train() if phase=='train' else model.eval()
            rloss = rcorr = 0; lbls = []; prds = []
            for inp, lbl in dataloaders[phase]:
                inp, lbl = inp.to(DEVICE), lbl.to(DEVICE)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase=='train'):
                    if phase=='train' and np.random.rand() < CUTMIX_PROB:
                        inp, ta, tb, lam = cutmix_data(inp, lbl, CUTMIX_ALPHA)
                        out  = model(inp)
                        loss = cutmix_criterion(criterion, out, ta, tb, lam)
                    else:
                        out  = model(inp)
                        loss = criterion(out, lbl)
                    _, pred = torch.max(out, 1)
                    if phase=='train': loss.backward(); optimizer.step()
                rloss += loss.item()*inp.size(0)
                rcorr += torch.sum(pred==lbl.data)
                lbls.extend(lbl.detach().cpu().numpy())
                prds.extend(pred.detach().cpu().numpy())

            eloss  = rloss / dataset_sizes[phase]
            eacc   = rcorr.double() / dataset_sizes[phase]
            ef1    = f1_score(lbls, prds, average='macro')
            print(f'{phase} Loss:{eloss:.4f} Acc:{eacc:.4f} F1:{ef1:.4f}')
            history[f'{phase}_loss'].append(eloss)
            history[f'{phase}_acc'].append(eacc.item())
            history[f'{phase}_macro_f1'].append(ef1)

            if phase=='val':
                scheduler.step(ef1)
                history['lr'].append(optimizer.param_groups[0]['lr'])
                if eacc > best_acc: best_acc = eacc
                if ef1  > best_val_macro_f1: best_val_macro_f1 = ef1
                if eloss < best_val_loss: best_val_loss = eloss
                if ef1 > best_metric:
                    best_metric = ef1; no_improve = 0
                    with open(BEST_MODEL_PATH,'wb') as f: torch.save(model.state_dict(), f)
                    print(f'En Iyi (F1:{best_metric:.4f}) kaydedildi.')
                else:
                    no_improve += 1
                    print(f'Iyilesme yok: {no_improve}/{EARLY_STOPPING_PATIENCE}')
        if no_improve >= EARLY_STOPPING_PATIENCE:
            print('Erken durdurma.'); break

    elapsed = time.time()-since
    print(f'Tamamlandi: {elapsed//60:.0f}dk {elapsed%60:.0f}sn | BestAcc:{best_acc:.4f} BestF1:{best_val_macro_f1:.4f}')
    with open(BEST_MODEL_PATH,'rb') as f: model.load_state_dict(torch.load(f, map_location=DEVICE))
    return model, history


model, history = train_model(model, criterion, optimizer, scheduler, EPOCHS)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pairs = [('train_acc','val_acc','Accuracy'),('train_loss','val_loss','Loss'),('train_macro_f1','val_macro_f1','Macro F1')]
for ax, (tr, vl, ylabel) in zip(axes, pairs):
    ax.plot(history.get(tr,[]), label='Train'); ax.plot(history.get(vl,[]), label='Val')
    ax.set_title(f'{MODEL_NAME} {ylabel} (CutMix+Rot)'); ax.set_xlabel('Epochs')
    ax.set_ylabel(ylabel); ax.legend(); ax.grid(True)
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, f'training_graph_{MODEL_NAME}_{RUN_TAG}.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight'); plt.show()
print(f'Grafik kaydedildi: {plot_path}')

In [ ]:
print('\nTEST SETI DEGERLENDIRMESI')
model.eval(); y_true=[]; y_pred=[]
with torch.no_grad():
    for inp, lbl in dataloaders['test']:
        out = model(inp.to(DEVICE)); _, p = torch.max(out,1)
        y_true.extend(lbl.cpu().numpy()); y_pred.extend(p.cpu().numpy())

report   = classification_report(y_true, y_pred, target_names=class_names, digits=4)
macro_f1 = f1_score(y_true, y_pred, average='macro')
print(report); print(f'Macro F1: {macro_f1:.4f}')

with open(os.path.join(REPORTS_DIR,f'classification_report_{MODEL_NAME}_{RUN_TAG}.txt'),'w',encoding='utf-8') as f:
    f.write(report + f'\nMacro F1: {macro_f1:.6f}\n')

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix - {MODEL_NAME} (CutMix+Rot)')
plt.xlabel('Tahmin'); plt.ylabel('Gercek')
plt.savefig(os.path.join(PLOTS_DIR,f'confusion_matrix_{MODEL_NAME}_{RUN_TAG}.png'), dpi=150, bbox_inches='tight')
plt.show()

n2i = {n:i for i,n in enumerate(class_names)}
p1 = p2 = None
if 'esophagitis' in n2i and 'normal-z-line' in n2i:
    i,j = n2i['esophagitis'], n2i['normal-z-line']; p1 = int(cm[i,j]+cm[j,i])
if 'dyed-lifted-polyps' in n2i and 'dyed-resection-margins' in n2i:
    i,j = n2i['dyed-lifted-polyps'], n2i['dyed-resection-margins']; p2 = int(cm[i,j]+cm[j,i])

latest = {'run_name':RUN_NAME,'run_tag':RUN_TAG,'model_name':MODEL_NAME,
          'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,
          'dropout_rate':DROPOUT_RATE,'label_smoothing':LABEL_SMOOTHING,'input_size':INPUT_SIZE,
          'cutmix_alpha':CUTMIX_ALPHA,'cutmix_prob':CUTMIX_PROB,
          'force_weighted_sampler':FORCE_WEIGHTED_SAMPLER,'use_weighted_sampler':False,
          'use_class_weighted_loss':USE_CLASS_WEIGHTED_LOSS,
          'early_stopping_patience':EARLY_STOPPING_PATIENCE,'early_stopping_metric':EARLY_STOPPING_METRIC,
          'best_val_acc':float(max(history['val_acc'])),'best_val_loss':float(min(history['val_loss'])),
          'best_val_macro_f1':float(max(history['val_macro_f1'])),'test_macro_f1':float(macro_f1),
          'pair_err_esophagitis_normal_z_line':p1,'pair_err_dyed_lifted_vs_resection':p2}

csv_path = os.path.join(RESULT_OUTPUT_DIR, f'ablation_results_{MODEL_NAME}.csv')
new_df   = pd.DataFrame([latest])
if os.path.exists(csv_path):
    old = pd.read_csv(csv_path)
    if 'run_tag' in old.columns: old = old[old['run_tag'] != RUN_TAG]
    new_df = pd.concat([old, new_df], ignore_index=True)
new_df.to_csv(csv_path, index=False)
print(f'Kaydedildi: {csv_path}')